In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.path as mpath

# ──────────────────────────────────────────────
# COLOR PALETTE
# ──────────────────────────────────────────────
BG_COLOR = '#E8ECF1'
NAVY     = '#1B3A5C'
TEAL     = '#2EAB8B'
GOLD     = '#D4A843'
WHITE    = '#FFFFFF'
GRAY     = '#8899AA'

# ──────────────────────────────────────────────
# FIGURE SETUP
# ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8.5, 6.5))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# Improved box drawing function with explicit pad control
def draw_box(x, y, text, color, text_color=WHITE, width=0.18, height=0.1, pad=0.03, fontsize=12, fontweight='bold', alpha=1.0, edgecolor=None):
    if edgecolor is None:
        edgecolor = color
    box = patches.FancyBboxPatch((x - width/2, y - height/2), width, height, 
                                 boxstyle=f"round,pad={pad},rounding_size=0.05", 
                                 linewidth=1.5, edgecolor=edgecolor, facecolor=color, alpha=alpha, zorder=3)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', color=text_color, 
            fontsize=fontsize, fontweight=fontweight, zorder=4)
    # Return true bounding box coordinates (Left, Right, Bottom, Top)
    return x - width/2 - pad, x + width/2 + pad, y - height/2 - pad, y + height/2 + pad

# ──────────────────────────────────────────────
# DRAW NODES
# ──────────────────────────────────────────────
# 1. Question Node
q_left, q_right, q_bot, q_top = draw_box(0.18, 0.85, "Question (Q)", NAVY, width=0.20, height=0.08, pad=0.04)

# 2. Context Group
# Background dashed box spanning the segments
ctx_box = patches.FancyBboxPatch((0.41, 0.79), 0.48, 0.12, 
                                 boxstyle="round,pad=0.03,rounding_size=0.05", 
                                 linewidth=2.0, edgecolor=GRAY, facecolor='none', linestyle='--', zorder=1)
ax.add_patch(ctx_box)
ax.text(0.65, 0.98, "Context Segments (Bandit Arms)", ha='center', va='center', color=NAVY, fontsize=13, fontweight='bold')

# Individual context chunks (Pad reduced to 0.01 to eliminate overlap)
draw_box(0.45, 0.85, "$s_1$", GRAY, width=0.06, height=0.08, pad=0.01, alpha=0.5)
s2_left, s2_right, s2_bot, s2_top = draw_box(0.55, 0.85, "$s_2$", GOLD, text_color=NAVY, width=0.06, height=0.08, pad=0.01)
draw_box(0.65, 0.85, "$s_3$", GRAY, width=0.06, height=0.08, pad=0.01, alpha=0.5)
ax.text(0.75, 0.85, "...", ha='center', va='center', color=NAVY, fontsize=16, fontweight='bold')
draw_box(0.85, 0.85, "$s_n$", GRAY, width=0.06, height=0.08, pad=0.01, alpha=0.5)

# 3. LLM API
llm_left, llm_right, llm_bot, llm_top = draw_box(0.55, 0.50, "Black-Box LLM API", NAVY, width=0.50, height=0.12, pad=0.04, fontsize=16)

# 4. Response Node
r_left, r_right, r_bot, r_top = draw_box(0.55, 0.15, "Response (R)", TEAL, width=0.25, height=0.10, pad=0.04, fontsize=14)

# ──────────────────────────────────────────────
# DRAW STANDARD ARROWS
# ──────────────────────────────────────────────
arrow_kwargs = dict(arrowstyle="-|>", mutation_scale=20, color=NAVY, lw=2.5, zorder=2)
# Q -> LLM
ax.annotate("", xy=(0.35, llm_top + 0.02), xytext=(0.18, q_bot - 0.02), arrowprops=dict(**arrow_kwargs, connectionstyle="arc3,rad=0.15"))
# Context -> LLM
ax.annotate("", xy=(0.65, llm_top + 0.02), xytext=(0.65, 0.75), arrowprops=arrow_kwargs)
# LLM -> Response
ax.annotate("", xy=(0.55, r_top + 0.02), xytext=(0.55, llm_bot - 0.02), arrowprops=arrow_kwargs)

# ──────────────────────────────────────────────
# GOLD ATTRIBUTION ARROW (Explicit Bézier Path)
# ──────────────────────────────────────────────
Path = mpath.Path
bezier_verts = [
    (r_left, 0.15),          # Start at the left edge of Response
    (0.08, 0.15),            # Control Pt 1: Pull far left
    (0.08, 0.75),            # Control Pt 2: Pull up, staying far left of LLM
    (s2_left + 0.01, s2_bot) # End: Curve into the bottom left of s2
]
bezier_codes = [Path.MOVETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
bezier_path = Path(bezier_verts, bezier_codes)

# FancyArrowPatch handles drawing the line + arrowhead along the defined path
gold_arrow = patches.FancyArrowPatch(path=bezier_path, arrowstyle="-|>", mutation_scale=22, 
                                     color=GOLD, lw=3, ls='--', zorder=5)
ax.add_patch(gold_arrow)

# Text box anchored to the vertical part of the curve
ax.text(0.18, 0.45, "Attribution:\nWhich arms\nmaximized reward?", 
        ha='center', va='center', color=GOLD, fontsize=14, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=WHITE, edgecolor=GOLD, lw=2, alpha=0.95), zorder=6)

plt.tight_layout()
plt.savefig('saved/camab_problem_diagram_fixed.svg', format='svg', bbox_inches='tight', facecolor=BG_COLOR)
plt.savefig('saved/camab_problem_diagram_fixed.png', dpi=300, bbox_inches='tight', facecolor=BG_COLOR)
print("Plot saved successfully.")

Plot saved successfully.
